# DMRG 05: Advanced Observables

This notebook demonstrates advanced measurements that are often used after a DMRG ground-state calculation: structure factors, finite-chain correlation-length estimates, entanglement spectra, string order parameters, ground-state fidelity, and many-body polarization.


## Measurement strategy

The current implementation uses dense contractions as a correctness baseline. This is appropriate for small chains and tutorial workflows. The API is written so these internals can later be replaced by environment-based MPS/MPO contractions for larger systems.


In [1]:
from pathlib import Path
import importlib
import shutil
import subprocess
import sys
import types


def _ensure_ipython_display_if_missing():
    try:
        import IPython.display  # noqa: F401
        return
    except ModuleNotFoundError:
        pass

    ipython_module = types.ModuleType("IPython")
    display_module = types.ModuleType("IPython.display")
    display_module.display = lambda *args, **kwargs: None
    display_module.Math = lambda *args, **kwargs: None
    ipython_module.display = display_module
    ipython_module.get_ipython = lambda: None
    ipython_module.version_info = (0, 0)
    sys.modules.setdefault("IPython", ipython_module)
    sys.modules.setdefault("IPython.display", display_module)


def _import_quantum_simulation():
    _ensure_ipython_display_if_missing()
    return importlib.import_module("quantum_simulation")


try:
    qs = _import_quantum_simulation()
except ModuleNotFoundError as exc:
    if exc.name != "quantum_simulation":
        raise
    repo_dir = Path("quantum-simulation")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/ToelUl/quantum-simulation.git"], check=True)
    target = Path("quantum_simulation")
    if not target.exists():
        shutil.copytree(repo_dir / "quantum_simulation", target)
    qs = _import_quantum_simulation()

print("Loaded quantum_simulation from", Path(qs.__file__).parent)


Loaded quantum_simulation from /mnt/d/phd_research_projects/quantum-simulation-main/quantum_simulation


In [2]:
import torch
import quantum_simulation as qs

from quantum_simulation import (
    DMRG,
    Ising1DMPOBuilder,
    MPS,
    Sweeps,
    correlation_length,
    entanglement_spectrum,
    ground_state_energy,
    ground_state_fidelity,
    many_body_polarization,
    string_order_parameter,
    structure_factor,
)

device = "cpu"  # Change to "cuda" after confirming your GPU setup.
dtype = torch.complex128
print(f"Using device={device}, dtype={dtype}")


Using device=cpu, dtype=torch.complex128


## Helper: run a small Ising DMRG calculation

The helper below keeps the notebook focused on measurements. It returns the optimized MPS, the MPO, and the DMRG energy trace for a transverse-field Ising chain.


In [3]:
def run_ising_dmrg(g_field, *, seed=2027):
    num_sites = 6
    builder = Ising1DMPOBuilder(
        num_sites=num_sites,
        j_coupling=1.0,
        g_field=g_field,
        device=device,
        dtype=dtype,
    )
    mpo = builder.get_mpo()
    initial_state = MPS(
        Nsites=num_sites,
        chid=2,
        initial_bond_dim=4,
        device=device,
        dtype=dtype,
        seed=seed,
    )
    sweeps = Sweeps(2)
    sweeps.set_schedule(
        maxdim=[4, 8],
        noise=[1e-6, 0.0],
        krylov_dim=[6, 8],
        cutoff=[1e-10, 1e-12],
        reortho=[False, True],
        maxreortho=[1, 1],
    )
    solver = DMRG(initial_state, mpo, device=device, dtype=dtype, contract_backend="einsum")
    energies, state = solver(sweeps, dispon=0, maxit=2)
    return state, mpo, energies, solver

state, mpo, energies, solver = run_ising_dmrg(g_field=0.7)
print(f"E0 = {ground_state_energy(energies):.12f}")
print(state)


Initializing MPS as a right-canonical random state...
Initialization complete. Ortho center at site 0.
[DMRG] Contraction backend resolved to 'einsum' on device='cpu'.
E0 = -2.327594419162
MPS(Nsites=6, chid=2, ortho_center=0, bond_dims=[2, 4, 6, 4, 2, 1])


## Local operators

The Ising MPO builders use spin-1/2 operators. We define `Sx`, `Sz`, a string unitary built from `Sz`, and a simple down-spin number operator for the polarization example.


In [4]:
Sx = torch.tensor([[0.0, 0.5], [0.5, 0.0]], device=device, dtype=dtype)
Sz = torch.tensor([[0.5, 0.0], [0.0, -0.5]], device=device, dtype=dtype)
string_unitary = torch.matrix_exp(1j * torch.pi * Sz)
number_down = torch.tensor([[0.0, 0.0], [0.0, 1.0]], device=device, dtype=dtype)
print("operators ready")


operators ready


## Structure factor

`S(q)` is the Fourier transform of the two-point correlation matrix. The dominant peak identifies the leading ordering wave vector for the chosen operator.


In [5]:
sf = structure_factor(state, Sx, connected=False)
q_values = sf["q_values"]
values = sf["values"].real
peak_idx = torch.argmax(values).item()

print("n   q/pi        Sx(q)")
for n, (q, value) in enumerate(zip(q_values, values)):
    print(f"{n:>1d}  {q.item() / torch.pi: .6f}  {value.item(): .10f}")
print(f"dominant q/pi = {q_values[peak_idx].item() / torch.pi:.6f}")


n   q/pi        Sx(q)
0   0.000000   0.5130545064
1   0.333333   0.2620400496
2   0.666667   0.1611068641
3   1.000000   0.1406516663
4   1.333333   0.1611068641
5   1.666667   0.2620400496
dominant q/pi = 0.000000


## Finite-chain correlation length

For a finite open chain, the helper estimates `xi` by fitting `log(|C(r)|)` to a straight line. This is a finite-size diagnostic, not an infinite-MPS transfer-matrix result.


In [6]:
xi_result = correlation_length(
    state,
    Sx,
    reference_site=0,
    fit_range=(1, 5),
    connected=True,
)
print(f"xi = {xi_result['xi']:.8f}")
print(f"slope = {xi_result['slope']:.8f}")
print("distance   C(r)")
for distance, corr in zip(xi_result["distances"], xi_result["correlations"]):
    print(f"{int(distance.item()):>8d}  {corr.real.item(): .10f}")


xi = 1.80097898
slope = -0.55525357
distance   C(r)
       1   0.0894911092
       2   0.0473205673
       3   0.0276641962
       4   0.0166551045
       5   0.0093931959


## Entanglement spectrum

The entanglement spectrum is `-log(lambda_alpha^2)` from the Schmidt values on a bond. Degeneracies or near-degeneracies can diagnose symmetry-protected or topological structure in suitable models.


In [7]:
center_bond = state.Nsites // 2
spectrum = entanglement_spectrum(state, bond=center_bond)
print(f"center bond = {center_bond}")
for level, value in enumerate(spectrum):
    print(f"level {level:>2d}: {value.item(): .10f}")


center bond = 3
level  0:  0.0419089533
level  1:  3.1941196606
level  2:  10.1609033547
level  3:  13.3131140646
level  4:  19.7183200004
level  5:  22.8705307176


## String order parameter

The generic string-order API measures `O_left(i) prod O_string(k) O_right(j)`. Below we use `Sz` endpoints and an `exp(i pi Sz)` string operator as a compact example.


In [8]:
string_value = string_order_parameter(
    state,
    Sz,
    string_unitary,
    Sz,
    0,
    state.Nsites - 1,
)
print(f"string order = {string_value.real.item():.12f} + {string_value.imag.item():.3e}j")


string order = 0.250000000000 + -5.863e-16j


## Ground-state fidelity

Fidelity compares two ground states at nearby Hamiltonian parameters. A sharp drop can indicate a quantum phase transition or a strong state rearrangement.


In [11]:
state_shifted, _, energies_shifted, _ = run_ising_dmrg(g_field=0.75)
fidelity = ground_state_fidelity(state, state_shifted)
print(f"E0(g=0.70) = {ground_state_energy(energies):.12f}")
print(f"E0(g=0.75) = {ground_state_energy(energies_shifted):.12f}")
print(f"F = {fidelity.item():.12f}")


Initializing MPS as a right-canonical random state...
Initialization complete. Ortho center at site 0.
[DMRG] Contraction backend resolved to 'einsum' on device='cpu'.
E0(g=0.70) = -2.327594419162
E0(g=0.75) = -2.461892867788
F = 0.999498463870


## Many-body polarization

The polarization helper evaluates the Resta unitary. The `origin` argument fixes the site-coordinate convention in the phase factor.


In [12]:
polarization = many_body_polarization(state, number_down, origin=0)
print(f"<U> = {polarization['expectation'].real.item():.10f} + {polarization['expectation'].imag.item():.10f}j")
print(f"phase = {polarization['phase'].item():.10f}")
print(f"P = {polarization['polarization'].item():.10f}")


<U> = 0.8230680516 + 0.0253198608j
phase = 0.0307530807
P = 0.0048945048


## Takeaways

The advanced measurement API exposes standard DMRG post-processing tools while keeping finite-size conventions explicit. The dense implementation is useful for correctness checks and small examples; optimized MPS environment contractions can be added later without changing these calls.
